In [1]:
import pandas as pd
import os

# This gets the project root regardless of where the notebook is running from
# os.path.abspath jumps up one level from the notebooks/ folder
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PATH = os.path.join(PROJECT_ROOT, "data", "10k_synthea_covid19_csv")

# Quick sanity check — should print your project root path
print("Project root:", PROJECT_ROOT)
print("Looking for data in:", DATA_PATH)

# Load the two most important tables first
patients = pd.read_csv(os.path.join(DATA_PATH, "patients.csv"))
encounters = pd.read_csv(os.path.join(DATA_PATH, "encounters.csv"))

print("\nPatients shape:", patients.shape)
print("Encounters shape:", encounters.shape)

print("\n--- Patients columns ---")
print(patients.columns.tolist())

print("\n--- Encounters columns ---")
print(encounters.columns.tolist())

Project root: C:\Users\brako\readmission-risk
Looking for data in: C:\Users\brako\readmission-risk\data\10k_synthea_covid19_csv

Patients shape: (12352, 25)
Encounters shape: (321528, 15)

--- Patients columns ---
['Id', 'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX', 'FIRST', 'LAST', 'SUFFIX', 'MAIDEN', 'MARITAL', 'RACE', 'ETHNICITY', 'GENDER', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE', 'COUNTY', 'ZIP', 'LAT', 'LON', 'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE']

--- Encounters columns ---
['Id', 'START', 'STOP', 'PATIENT', 'ORGANIZATION', 'PROVIDER', 'PAYER', 'ENCOUNTERCLASS', 'CODE', 'DESCRIPTION', 'BASE_ENCOUNTER_COST', 'TOTAL_CLAIM_COST', 'PAYER_COVERAGE', 'REASONCODE', 'REASONDESCRIPTION']


In [2]:
import pandas as pd
import os

# --- Encounter class distribution ---
# This tells us what types of visits exist
print("=== ENCOUNTER CLASSES ===")
print(encounters['ENCOUNTERCLASS'].value_counts())

print("\n=== ENCOUNTER DESCRIPTIONS (top 20) ===")
print(encounters['DESCRIPTION'].value_counts().head(20))

print("\n=== DATE RANGE ===")
# Convert to datetime so we can work with them properly
encounters['START'] = pd.to_datetime(encounters['START'])
encounters['STOP'] = pd.to_datetime(encounters['STOP'])
print("Earliest encounter:", encounters['START'].min())
print("Latest encounter:", encounters['START'].max())

print("\n=== MISSING VALUES - PATIENTS ===")
print(patients.isnull().sum()[patients.isnull().sum() > 0])

print("\n=== MISSING VALUES - ENCOUNTERS ===")
print(encounters.isnull().sum()[encounters.isnull().sum() > 0])

print("\n=== PATIENTS DECEASED ===")
print("Deceased:", patients['DEATHDATE'].notna().sum())
print("Alive:", patients['DEATHDATE'].isna().sum())

=== ENCOUNTER CLASSES ===
ENCOUNTERCLASS
wellness      156219
ambulatory     86069
outpatient     58367
inpatient       9584
emergency       7233
urgentcare      4056
Name: count, dtype: int64

=== ENCOUNTER DESCRIPTIONS (top 20) ===
DESCRIPTION
General examination of patient (procedure)      141988
Encounter for check up (procedure)               50829
Follow-up encounter                              26363
Encounter for symptom                            13599
Encounter for problem                            13250
Well child visit (procedure)                     12461
Encounter for problem (procedure)                10062
Encounter for symptom (procedure)                 9295
Prenatal initial visit                            4427
Urgent care clinic (procedure)                    4056
Prenatal visit                                    3593
Encounter Inpatient                               2792
Consultation for treatment                        2780
Outpatient procedure                   

In [3]:
# --- Find COVID-related encounters ---
# We can't rely on REASONDESCRIPTION alone (78% missing)
# So we search across DESCRIPTION and REASONDESCRIPTION both

covid_keywords = [
    'covid', 'COVID', 'coronavirus', 'SARS', 'isolation'
]

# Build a mask that checks both description columns
desc_mask = encounters['DESCRIPTION'].str.contains(
    '|'.join(covid_keywords), case=False, na=False
)
reason_mask = encounters['REASONDESCRIPTION'].str.contains(
    '|'.join(covid_keywords), case=False, na=False
)

covid_encounters = encounters[desc_mask | reason_mask].copy()

print("=== COVID ENCOUNTER VOLUME ===")
print("Total COVID-related encounters:", len(covid_encounters))
print("\nBy encounter class:")
print(covid_encounters['ENCOUNTERCLASS'].value_counts())

print("\nBy description (top 15):")
print(covid_encounters['DESCRIPTION'].value_counts().head(15))

print("\nUnique patients with COVID encounters:")
print(covid_encounters['PATIENT'].nunique())

print("\nDate range of COVID encounters:")
print("Earliest:", covid_encounters['START'].min())
print("Latest:", covid_encounters['START'].max())

=== COVID ENCOUNTER VOLUME ===
Total COVID-related encounters: 2721

By encounter class:
ENCOUNTERCLASS
inpatient    2376
wellness      345
Name: count, dtype: int64

By description (top 15):
DESCRIPTION
Hospital admission for isolation (procedure)       1867
Admission to intensive care unit (procedure)        375
Death Certification                                 345
Hospital admission  for observation (procedure)     134
Name: count, dtype: int64

Unique patients with COVID encounters:
1867

Date range of COVID encounters:
Earliest: 2020-01-26 18:55:26+00:00
Latest: 2020-04-15 02:17:11+00:00


In [4]:
# =============================================================
# COHORT CONSTRUCTION
# Goal: Define one row per patient representing their
#       index COVID inpatient encounter
# =============================================================

# Step 1 — Isolate only inpatient COVID encounters
# We exclude 'wellness' class which captured death certifications
covid_inpatient = covid_encounters[
    covid_encounters['ENCOUNTERCLASS'] == 'inpatient'
].copy()

print("Inpatient COVID encounters:", len(covid_inpatient))

# Step 2 — For each patient, take their FIRST inpatient COVID encounter
# This is the "index encounter" — the starting event for our analysis
# Some patients may have multiple inpatient stays; we anchor on the first
index_encounters = (
    covid_inpatient
    .sort_values('START')
    .groupby('PATIENT')
    .first()
    .reset_index()
)

print("Unique patients (index encounters):", len(index_encounters))

# Step 3 — Identify patients who died DURING their index admission
# If STOP is null or patient died before discharge, exclude them
# A patient who died in hospital was never discharged — can't be readmitted
index_encounters['START'] = pd.to_datetime(index_encounters['START'])
index_encounters['STOP'] = pd.to_datetime(index_encounters['STOP'])

# Merge in death dates from patients table
index_encounters = index_encounters.merge(
    patients[['Id', 'DEATHDATE']],
    left_on='PATIENT',
    right_on='Id',
    how='left'
)

index_encounters['DEATHDATE'] = pd.to_datetime(
    index_encounters['DEATHDATE'], utc=True
)

# Flag patients who died during or before discharge
# i.e. death date is before or on the same day as encounter STOP
died_during_admission = (
    index_encounters['DEATHDATE'].notna() &
    (index_encounters['DEATHDATE'] <= index_encounters['STOP'])
)

print("\nPatients who died during index admission:", died_during_admission.sum())

# Step 4 — Exclude those patients from our cohort
cohort = index_encounters[~died_during_admission].copy()
print("Final cohort size:", len(cohort))

# Step 5 — Calculate length of stay in days
# This becomes a feature later — longer stays = sicker patients
cohort['length_of_stay'] = (
    (cohort['STOP'] - cohort['START']).dt.total_seconds() / 86400
).round(2)

print("\nLength of stay (days):")
print(cohort['length_of_stay'].describe().round(2))

Inpatient COVID encounters: 2376
Unique patients (index encounters): 1867

Patients who died during index admission: 101
Final cohort size: 1766

Length of stay (days):
count    1766.00
mean       13.87
std         4.38
min         1.00
25%        11.29
50%        14.18
75%        17.23
max        21.49
Name: length_of_stay, dtype: float64


In [5]:
# =============================================================
# TARGET VARIABLE CONSTRUCTION
# Goal: Label each patient 1 if readmitted within 30 days,
#       0 if not
# This is called the "outcome" or "target" in ML terms
# In clinical terms this is the "endpoint"
# =============================================================

# Step 1 — Get all future encounters for cohort patients
# We need every encounter AFTER the index discharge
cohort_patient_ids = cohort['PATIENT'].unique()

# Filter all encounters to just our cohort patients
all_cohort_encounters = encounters[
    encounters['PATIENT'].isin(cohort_patient_ids)
].copy()

all_cohort_encounters['START'] = pd.to_datetime(
    all_cohort_encounters['START'], utc=True
)
all_cohort_encounters['STOP'] = pd.to_datetime(
    all_cohort_encounters['STOP'], utc=True
)

print("Total encounters for cohort patients:", len(all_cohort_encounters))

# Step 2 — For each patient, find encounters that started
# AFTER their index discharge and WITHIN 30 days
# We only count inpatient and emergency as true readmissions
readmission_classes = ['inpatient', 'emergency']

def check_readmission(row):
    """
    For a given index encounter, look for any inpatient or emergency
    encounter that started after discharge and within 30 days.
    
    This is the clinical definition of a readmission.
    """
    discharge_date = row['STOP']
    patient_id = row['PATIENT']
    window_end = discharge_date + pd.Timedelta(days=30)
    
    # Get this patient's subsequent encounters
    subsequent = all_cohort_encounters[
        (all_cohort_encounters['PATIENT'] == patient_id) &
        (all_cohort_encounters['START'] > discharge_date) &
        (all_cohort_encounters['START'] <= window_end) &
        (all_cohort_encounters['ENCOUNTERCLASS'].isin(readmission_classes))
    ]
    
    return 1 if len(subsequent) > 0 else 0

# Apply to every row in cohort
# Note: this may take 30-60 seconds on 1,766 rows
print("Calculating readmission labels... (this may take a moment)")
cohort['readmitted_30d'] = cohort.apply(check_readmission, axis=1)

print("\n=== TARGET VARIABLE DISTRIBUTION ===")
print(cohort['readmitted_30d'].value_counts())
print("\nPositive rate (readmitted):",
      round(cohort['readmitted_30d'].mean() * 100, 2), "%")

Total encounters for cohort patients: 65410
Calculating readmission labels... (this may take a moment)

=== TARGET VARIABLE DISTRIBUTION ===
readmitted_30d
0    1596
1     170
Name: count, dtype: int64

Positive rate (readmitted): 9.63 %


In [6]:
# =============================================================
# PHASE 2 — FEATURE ENGINEERING
# Domain 1: Demographics
# Source: patients.csv
# =============================================================

# Step 1 — Load patients and select only clinically relevant columns
# We immediately drop PHI — SSN, drivers license, passport
# This is both a privacy requirement and good modeling practice
# PHI fields carry no predictive signal for readmission

demographic_cols = [
    'Id',
    'BIRTHDATE',
    'DEATHDATE',    # keeping temporarily to calculate age
    'GENDER',
    'RACE',
    'ETHNICITY',
    'MARITAL',
    'ZIP',
    'HEALTHCARE_EXPENSES',
    'HEALTHCARE_COVERAGE'
]

patients_clean = patients[demographic_cols].copy()

# Step 2 — Calculate age at time of index admission
# Age = a continuous variable is more useful than age buckets for ML
# We calculate from BIRTHDATE to the index encounter START date

# First merge discharge dates from cohort into patients
patients_clean = patients_clean.merge(
    cohort[['PATIENT', 'START', 'STOP', 'length_of_stay', 'readmitted_30d']],
    left_on='Id',
    right_on='PATIENT',
    how='inner'  # inner join — only keep patients in our cohort
)

# Convert BIRTHDATE to datetime
patients_clean['BIRTHDATE'] = pd.to_datetime(
    patients_clean['BIRTHDATE'], utc=True
)

# Calculate age in years at admission
patients_clean['age_at_admission'] = (
    (patients_clean['START'] - patients_clean['BIRTHDATE'])
    .dt.days / 365.25
).round(1)

print("=== AGE DISTRIBUTION ===")
print(patients_clean['age_at_admission'].describe().round(1))

print("\n=== GENDER DISTRIBUTION ===")
print(patients_clean['GENDER'].value_counts())

print("\n=== RACE DISTRIBUTION ===")
print(patients_clean['RACE'].value_counts())

print("\n=== MARITAL STATUS ===")
print(patients_clean['MARITAL'].value_counts(dropna=False))

print("\n=== MISSING VALUES ===")
print(patients_clean.isnull().sum()[patients_clean.isnull().sum() > 0])

print("\nCohort size after merge:", len(patients_clean))

=== AGE DISTRIBUTION ===
count    1766.0
mean       51.6
std        21.0
min         0.0
25%        36.6
50%        53.2
75%        65.2
max       110.1
Name: age_at_admission, dtype: float64

=== GENDER DISTRIBUTION ===
GENDER
F    963
M    803
Name: count, dtype: int64

=== RACE DISTRIBUTION ===
RACE
white     1475
black      162
asian      123
native       6
Name: count, dtype: int64

=== MARITAL STATUS ===
MARITAL
M      1212
S       297
NaN     257
Name: count, dtype: int64

=== MISSING VALUES ===
DEATHDATE    1518
MARITAL       257
ZIP           821
dtype: int64

Cohort size after merge: 1766


In [7]:
# =============================================================
# FEATURE ENGINEERING — Domain 1 continued
# Cleaning and encoding demographic features
# =============================================================

# Step 1 — Drop columns we no longer need or can't use
drop_cols = ['BIRTHDATE', 'DEATHDATE', 'ZIP', 'Id', 'PATIENT']
patients_clean = patients_clean.drop(columns=drop_cols)

# Step 2 — Handle missing marital status
# We treat missing as its own category — 'U' for unknown
# Why: unknown social support is clinically different from
# known married or known single — imputing the mode would
# hide that uncertainty from the model
patients_clean['MARITAL'] = patients_clean['MARITAL'].fillna('U')

# Step 3 — Encode categorical variables
# We use pd.get_dummies (one-hot encoding) for nominal categories
# drop_first=True avoids the dummy variable trap
# (multicollinearity issue where one category is perfectly
# predictable from the others)

patients_encoded = pd.get_dummies(
    patients_clean,
    columns=['GENDER', 'RACE', 'ETHNICITY', 'MARITAL'],
    drop_first=True,
    dtype=int   # store as 0/1 integers not booleans
)

print("=== ENCODED FEATURE COLUMNS ===")
print(patients_encoded.columns.tolist())

print("\nShape:", patients_encoded.shape)
print("\nFirst 3 rows:")
print(patients_encoded.head(3))

print("\n=== ANY REMAINING NULLS? ===")
print(patients_encoded.isnull().sum()[patients_encoded.isnull().sum() > 0])

=== ENCODED FEATURE COLUMNS ===
['HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE', 'START', 'STOP', 'length_of_stay', 'readmitted_30d', 'age_at_admission', 'GENDER_M', 'RACE_black', 'RACE_native', 'RACE_white', 'ETHNICITY_nonhispanic', 'MARITAL_S', 'MARITAL_U']

Shape: (1766, 14)

First 3 rows:
   HEALTHCARE_EXPENSES  HEALTHCARE_COVERAGE                     START  \
0             22940.00               893.28 2020-02-19 05:30:14+00:00   
1            792765.79              7389.24 2020-03-07 13:00:03+00:00   
2           1533497.00              7207.72 2020-03-15 00:32:17+00:00   

                       STOP  length_of_stay  readmitted_30d  age_at_admission  \
0 2020-03-05 14:56:14+00:00           15.39               0               0.7   
1 2020-03-28 17:29:03+00:00           21.19               0              36.4   
2 2020-04-02 06:25:17+00:00           18.25               0              60.7   

   GENDER_M  RACE_black  RACE_native  RACE_white  ETHNICITY_nonhispanic  \
0         0    

In [8]:
# =============================================================
# COHORT REFINEMENT — Exclude pediatric patients
# Inclusion criterion: Age >= 18 at time of admission
# Rationale: Pediatric COVID follows different clinical pathway
#            Readmission drivers are not generalizable to adults
# =============================================================

print("Cohort size before age filter:", len(cohort))

# Filter cohort to adults only
adult_mask = patients_encoded['age_at_admission'] >= 18

# Apply to both cohort and patients_encoded
# We need to keep them aligned — same patients, same order
cohort = cohort[adult_mask.values].reset_index(drop=True)
patients_encoded = patients_encoded[adult_mask].reset_index(drop=True)

print("Cohort size after age filter:", len(cohort))
print("Patients removed:", adult_mask.value_counts()[False])

print("\nAge distribution after filter:")
print(patients_encoded['age_at_admission'].describe().round(1))

Cohort size before age filter: 1766
Cohort size after age filter: 1662
Patients removed: 104

Age distribution after filter:
count    1662.0
mean       54.2
std        18.7
min        18.0
25%        40.4
50%        54.8
75%        66.1
max       110.1
Name: age_at_admission, dtype: float64


In [9]:
# =============================================================
# FEATURE ENGINEERING — Domain 2: Encounter Context
# Source: encounters.csv
# Question: How severe was this patient's index admission?
# All features derived from the index encounter only —
# nothing from after discharge (temporal leakage prevention)
# =============================================================

# Rebuild feature table from the filtered adult cohort
feature_table = cohort[['PATIENT', 'START', 'STOP',
                          'length_of_stay', 'readmitted_30d']].copy()

# --- Feature 1: ICU Admission Flag ---
# ICU admission = proxy for severe disease
# Source: covid_encounters description field
icu_patients = covid_encounters[
    covid_encounters['DESCRIPTION'].str.contains(
        'intensive care', case=False, na=False
    )
]['PATIENT'].unique()

feature_table['icu_admission'] = (
    feature_table['PATIENT'].isin(icu_patients).astype(int)
)

# --- Feature 2: Discharge Hour ---
# Late night discharge = less immediate access to follow-up care
feature_table['discharge_hour'] = feature_table['STOP'].dt.hour

# --- Feature 3: Weekend Discharge ---
# Weekend discharge = PCP offices closed, harder to arrange follow-up
# This is a real signal in readmission literature
# 5 = Saturday, 6 = Sunday
feature_table['weekend_discharge'] = (
    feature_table['STOP'].dt.dayofweek >= 5
).astype(int)

print("=== DOMAIN 2 FEATURES ===")
print("\nICU admission rate:",
      round(feature_table['icu_admission'].mean() * 100, 2), "%")
print("Weekend discharge rate:",
      round(feature_table['weekend_discharge'].mean() * 100, 2), "%")

print("\nDischarge hour distribution:")
print(feature_table['discharge_hour'].describe().round(1))

print("\nFeature table shape:", feature_table.shape)
print("\nFirst 3 rows:")
print(feature_table[['PATIENT', 'length_of_stay', 'icu_admission',
                       'discharge_hour', 'weekend_discharge',
                       'readmitted_30d']].head(3))

=== DOMAIN 2 FEATURES ===

ICU admission rate: 21.42 %
Weekend discharge rate: 29.24 %

Discharge hour distribution:
count    1662.0
mean       11.5
std         6.9
min         0.0
25%         6.0
50%        12.0
75%        17.0
max        23.0
Name: discharge_hour, dtype: float64

Feature table shape: (1662, 8)

First 3 rows:
                                PATIENT  length_of_stay  icu_admission  \
0  00093cdd-a9f0-4ad8-87e9-53534501f008           15.31              0   
1  002dd6c0-26c5-4bd0-b0f2-3b6f600b132f           14.16              0   
2  005a2319-85cd-40b6-b0bc-749cef080c7e           13.29              0   

   discharge_hour  weekend_discharge  readmitted_30d  
0               1                  0               0  
1              13                  0               0  
2              23                  1               0  


In [10]:
# =============================================================
# FEATURE ENGINEERING — Domain 3: Prior Utilization
# Source: encounters.csv
# Question: How heavy a healthcare user was this patient
#           in the 6 months BEFORE their index admission?
# Lookback window: 180 days prior to index START date
# =============================================================

# Define lookback window
LOOKBACK_DAYS = 180

def get_prior_utilization(patient_id, index_start):
    """
    For a given patient, count encounters in the 180 days
    before their index admission by encounter class.
    
    Parameters:
        patient_id: the patient UUID
        index_start: datetime of index admission start
    
    Returns:
        dict of counts by encounter type
    """
    window_start = index_start - pd.Timedelta(days=LOOKBACK_DAYS)
    
    # Get all encounters for this patient in the lookback window
    prior = all_cohort_encounters[
        (all_cohort_encounters['PATIENT'] == patient_id) &
        (all_cohort_encounters['START'] >= window_start) &
        (all_cohort_encounters['START'] < index_start)  # strictly before admission
    ]
    
    return {
        'prior_inpatient':  int((prior['ENCOUNTERCLASS'] == 'inpatient').sum()),
        'prior_emergency':  int((prior['ENCOUNTERCLASS'] == 'emergency').sum()),
        'prior_outpatient': int((prior['ENCOUNTERCLASS'].isin(
                                ['outpatient', 'ambulatory'])).sum()),
        'prior_total':      len(prior)
    }

# Apply to every patient in cohort
# This iterates 1,662 rows — may take 30-60 seconds
print("Calculating prior utilization... (may take a moment)")

utilization_records = []
for _, row in feature_table.iterrows():
    util = get_prior_utilization(row['PATIENT'], row['START'])
    util['PATIENT'] = row['PATIENT']
    utilization_records.append(util)

utilization_df = pd.DataFrame(utilization_records)

print("\n=== PRIOR UTILIZATION (6 months before admission) ===")
print(utilization_df[['prior_inpatient', 'prior_emergency',
                        'prior_outpatient', 'prior_total']].describe().round(2))

print("\nPatients with ANY prior inpatient stay:",
      (utilization_df['prior_inpatient'] > 0).sum())
print("Patients with ANY prior ED visit:",
      (utilization_df['prior_emergency'] > 0).sum())

# Merge into feature table
feature_table = feature_table.merge(utilization_df, on='PATIENT', how='left')
print("\nFeature table shape:", feature_table.shape)

Calculating prior utilization... (may take a moment)

=== PRIOR UTILIZATION (6 months before admission) ===
       prior_inpatient  prior_emergency  prior_outpatient  prior_total
count          1662.00          1662.00           1662.00      1662.00
mean              0.10             0.09              2.23         3.10
std               1.29             0.39              5.20         6.05
min               0.00             0.00              1.00         1.00
25%               0.00             0.00              1.00         1.00
50%               0.00             0.00              1.00         2.00
75%               0.00             0.00              2.00         3.00
max              36.00             9.00             72.00        89.00

Patients with ANY prior inpatient stay: 61
Patients with ANY prior ED visit: 124

Feature table shape: (1662, 12)


In [11]:
# =============================================================
# FEATURE ENGINEERING — Domain 4: Comorbidity Burden
# Source: conditions.csv
# Question: How many active chronic conditions did this
#           patient have AT THE TIME OF DISCHARGE?
#
# Temporal leakage check: a condition counts as "active at
# discharge" if it started before/during the index admission
# AND either has no stop date (ongoing) or stopped after discharge
# =============================================================

conditions = pd.read_csv(os.path.join(DATA_PATH, "conditions.csv"))

conditions['START'] = pd.to_datetime(conditions['START'], utc=True)
conditions['STOP'] = pd.to_datetime(conditions['STOP'], utc=True)

print("Conditions table shape:", conditions.shape)
print("\nTop 15 most common conditions:")
print(conditions['DESCRIPTION'].value_counts().head(15))

def count_active_conditions(patient_id, discharge_date):
    """
    Count conditions that were active at the time of discharge.
    A condition is active if:
      - it started on or before discharge_date, AND
      - it either has no STOP date (still ongoing) or
        STOP is after discharge_date
    """
    patient_conditions = conditions[conditions['PATIENT'] == patient_id]
    
    active = patient_conditions[
        (patient_conditions['START'] <= discharge_date) &
        (
            patient_conditions['STOP'].isna() |
            (patient_conditions['STOP'] > discharge_date)
        )
    ]
    
    return len(active)

print("\nCalculating active comorbidity counts... (may take a moment)")

feature_table['active_conditions'] = feature_table.apply(
    lambda row: count_active_conditions(row['PATIENT'], row['STOP']),
    axis=1
)

print("\n=== ACTIVE CONDITIONS AT DISCHARGE ===")
print(feature_table['active_conditions'].describe().round(2))

print("\nFeature table shape:", feature_table.shape)

Conditions table shape: (114544, 6)

Top 15 most common conditions:
DESCRIPTION
Suspected COVID-19                         9106
COVID-19                                   8820
Fever (finding)                            8083
Cough (finding)                            6202
Body mass index 30+ - obesity (finding)    5002
Loss of taste (finding)                    4711
Prediabetes                                3917
Anemia (disorder)                          3650
Fatigue (finding)                          3516
Hypertension                               3168
Sputum finding (finding)                   2970
Chronic sinusitis (disorder)               2655
Miscarriage in first trimester             2212
Pneumonia (disorder)                       1867
Hypoxemia (disorder)                       1867
Name: count, dtype: int64

Calculating active comorbidity counts... (may take a moment)

=== ACTIVE CONDITIONS AT DISCHARGE ===
count    1662.00
mean        8.12
std         6.97
min         1.00
25% 

In [12]:
# =============================================================
# FEATURE ENGINEERING — Domain 4 continued
# Specific Comorbidity Flags
# Source: conditions.csv
# Rationale: These 4 conditions have strong literature support
#            as COVID severity / readmission risk factors
# =============================================================

# Define keyword groups for each comorbidity category
comorbidity_keywords = {
    'has_hypertension':       ['hypertension'],
    'has_obesity':             ['obesity'],
    'has_diabetes':            ['diabetes', 'prediabetes'],
    'has_chronic_resp':        ['copd', 'chronic obstructive',
                                 'asthma', 'chronic sinusitis',
                                 'pulmonary disease']
}

def has_active_condition_matching(patient_id, discharge_date, keywords):
    """
    Check if patient has an ACTIVE condition at discharge
    matching any of the given keywords.
    
    Same temporal logic as count_active_conditions:
    - started on or before discharge
    - either ongoing (no STOP) or STOP after discharge
    """
    patient_conditions = conditions[conditions['PATIENT'] == patient_id]
    
    active = patient_conditions[
        (patient_conditions['START'] <= discharge_date) &
        (
            patient_conditions['STOP'].isna() |
            (patient_conditions['STOP'] > discharge_date)
        )
    ]
    
    # Check if any active condition description matches our keywords
    pattern = '|'.join(keywords)
    match = active['DESCRIPTION'].str.contains(pattern, case=False, na=False)
    
    return int(match.any())

print("Calculating comorbidity flags... (may take a moment - 4 passes)")

for flag_name, keywords in comorbidity_keywords.items():
    feature_table[flag_name] = feature_table.apply(
        lambda row: has_active_condition_matching(
            row['PATIENT'], row['STOP'], keywords
        ),
        axis=1
    )
    prevalence = feature_table[flag_name].mean() * 100
    print(f"{flag_name}: {prevalence:.1f}% of cohort")

print("\nFeature table shape:", feature_table.shape)
print("\n=== COMORBIDITY FLAG SUMMARY ===")
print(feature_table[list(comorbidity_keywords.keys())].sum())

Calculating comorbidity flags... (may take a moment - 4 passes)
has_hypertension: 40.3% of cohort
has_obesity: 47.4% of cohort
has_diabetes: 45.2% of cohort
has_chronic_resp: 30.7% of cohort

Feature table shape: (1662, 17)

=== COMORBIDITY FLAG SUMMARY ===
has_hypertension    670
has_obesity         787
has_diabetes        751
has_chronic_resp    510
dtype: int64


In [13]:
# =============================================================
# FEATURE ENGINEERING — Domain 5: Medication Burden
# Source: medications.csv
# Question: How many active medications did this patient
#           have AT THE TIME OF DISCHARGE?
# Same temporal logic as Domain 4 — active at discharge means
# started on/before discharge AND (no stop date OR stop after discharge)
# =============================================================

medications = pd.read_csv(os.path.join(DATA_PATH, "medications.csv"))

medications['START'] = pd.to_datetime(medications['START'], utc=True)
medications['STOP'] = pd.to_datetime(medications['STOP'], utc=True)

print("Medications table shape:", medications.shape)
print("\nTop 10 most common medications:")
print(medications['DESCRIPTION'].value_counts().head(10))

def count_active_medications(patient_id, discharge_date):
    """
    Count medications active at time of discharge.
    Same active-window logic as conditions.
    """
    patient_meds = medications[medications['PATIENT'] == patient_id]
    
    active = patient_meds[
        (patient_meds['START'] <= discharge_date) &
        (
            patient_meds['STOP'].isna() |
            (patient_meds['STOP'] > discharge_date)
        )
    ]
    
    return len(active)

print("\nCalculating active medication counts... (may take a moment)")

feature_table['active_medications'] = feature_table.apply(
    lambda row: count_active_medications(row['PATIENT'], row['STOP']),
    axis=1
)

# Polypharmacy flag — 5+ active medications is the standard clinical threshold
feature_table['polypharmacy'] = (
    feature_table['active_medications'] >= 5
).astype(int)

print("\n=== ACTIVE MEDICATIONS AT DISCHARGE ===")
print(feature_table['active_medications'].describe().round(2))

print("\nPolypharmacy rate (5+ meds):",
      round(feature_table['polypharmacy'].mean() * 100, 2), "%")

print("\nFeature table shape:", feature_table.shape)

Medications table shape: (431262, 13)

Top 10 most common medications:
DESCRIPTION
Hydrochlorothiazide 25 MG Oral Tablet                                                                   53716
insulin human  isophane 70 UNT/ML / Regular Insulin  Human 30 UNT/ML Injectable Suspension [Humulin]    52201
amLODIPine 5 MG / Hydrochlorothiazide 12.5 MG / Olmesartan medoxomil 20 MG Oral Tablet                  36464
Atenolol 50 MG / Chlorthalidone 25 MG Oral Tablet                                                       36170
24 HR Metformin hydrochloride 500 MG Extended Release Oral Tablet                                       32633
Simvastatin 10 MG Oral Tablet                                                                           26313
NDA020503 200 ACTUAT Albuterol 0.09 MG/ACTUAT Metered Dose Inhaler                                      25684
120 ACTUAT Fluticasone propionate 0.044 MG/ACTUAT Metered Dose Inhaler                                  24596
Hydrochlorothiazide 12.5 MG          

In [14]:
# =============================================================
# FINAL ASSEMBLY — Combine all domains into modeling dataset
# =============================================================

# Step 1 — Merge Domain 1 demographics into feature_table
# patients_encoded has: HEALTHCARE_EXPENSES, HEALTHCARE_COVERAGE,
#   age_at_admission, GENDER_M, RACE_*, ETHNICITY_*, MARITAL_*
# We merge on PATIENT (feature_table) <-> what was originally 'Id'/'PATIENT'

demographic_features = patients_encoded[[
    'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE', 'age_at_admission',
    'GENDER_M', 'RACE_black', 'RACE_native', 'RACE_white',
    'ETHNICITY_nonhispanic', 'MARITAL_S', 'MARITAL_U'
]].copy()

# Re-attach PATIENT id from cohort for merging (same row order as cohort)
demographic_features['PATIENT'] = cohort['PATIENT'].values

# Merge into feature_table
model_df = feature_table.merge(demographic_features, on='PATIENT', how='left')

print("After merging demographics:", model_df.shape)

# Step 2 — Drop columns not needed for modeling
# PATIENT, START, STOP — identifiers/timestamps, not features
# (We'll save PATIENT separately for traceability)
patient_ids = model_df['PATIENT'].copy()  # keep for reference

model_df = model_df.drop(columns=['PATIENT', 'START', 'STOP'])

print("Final modeling dataset shape:", model_df.shape)
print("\nColumns:")
print(model_df.columns.tolist())

print("\n=== TARGET VARIABLE CHECK ===")
print(model_df['readmitted_30d'].value_counts())
print("Positive rate:", round(model_df['readmitted_30d'].mean() * 100, 2), "%")

print("\n=== ANY MISSING VALUES? ===")
print(model_df.isnull().sum()[model_df.isnull().sum() > 0])

print("\n=== FINAL DATASET PREVIEW ===")
print(model_df.head(3))

After merging demographics: (1662, 29)
Final modeling dataset shape: (1662, 26)

Columns:
['length_of_stay', 'readmitted_30d', 'icu_admission', 'discharge_hour', 'weekend_discharge', 'prior_inpatient', 'prior_emergency', 'prior_outpatient', 'prior_total', 'active_conditions', 'has_hypertension', 'has_obesity', 'has_diabetes', 'has_chronic_resp', 'active_medications', 'polypharmacy', 'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE', 'age_at_admission', 'GENDER_M', 'RACE_black', 'RACE_native', 'RACE_white', 'ETHNICITY_nonhispanic', 'MARITAL_S', 'MARITAL_U']

=== TARGET VARIABLE CHECK ===
readmitted_30d
0    1504
1     158
Name: count, dtype: int64
Positive rate: 9.51 %

=== ANY MISSING VALUES? ===
Series([], dtype: int64)

=== FINAL DATASET PREVIEW ===
   length_of_stay  readmitted_30d  icu_admission  discharge_hour  \
0           15.31               0              0               1   
1           14.16               0              0              13   
2           13.29               0      

In [16]:
# =============================================================
# SAVE MODELING-READY DATASET
# =============================================================

import os

# Create processed data directory if it doesn't exist
processed_dir = os.path.join(PROJECT_ROOT, "data", "processed")
os.makedirs(processed_dir, exist_ok=True)

# Save the modeling dataset
output_path = os.path.join(processed_dir, "model_dataset.csv")
model_df.to_csv(output_path, index=False)

# Also save patient_ids separately for traceability
# (never include in model_df, but keep for linking predictions back)
ids_path = os.path.join(processed_dir, "patient_ids.csv")
patient_ids.to_csv(ids_path, index=False)

print(f"Saved model dataset to: {output_path}")
print(f"Saved patient IDs to: {ids_path}")
print(f"\nModel dataset shape: {model_df.shape}")

Saved model dataset to: C:\Users\brako\readmission-risk\data\processed\model_dataset.csv
Saved patient IDs to: C:\Users\brako\readmission-risk\data\processed\patient_ids.csv

Model dataset shape: (1662, 26)


In [ ]:
# =============================================================
# SUBGROUP ANALYSIS — Gender and Race
# Purpose: Check for disparities in readmission rates and
# key clinical features BEFORE modeling.
# This is standard practice in health equity-focused analytics.
# =============================================================

import pandas as pd

# Reload race/gender as readable labels (not yet one-hot encoded)
# We pull from patients_clean (pre-encoding) aligned to our final cohort
demo_raw = patients[['Id', 'GENDER', 'RACE']].copy()
demo_raw = demo_raw.merge(
    pd.DataFrame({'PATIENT': patient_ids.values}),
    left_on='Id', right_on='PATIENT', how='inner'
)

# Attach target and key features
analysis_df = demo_raw[['PATIENT', 'GENDER', 'RACE']].copy()
analysis_df['readmitted_30d'] = model_df['readmitted_30d'].values
analysis_df['age_at_admission'] = model_df['age_at_admission'].values
analysis_df['icu_admission'] = model_df['icu_admission'].values
analysis_df['active_conditions'] = model_df['active_conditions'].values
analysis_df['length_of_stay'] = model_df['length_of_stay'].values

print("=== READMISSION RATE BY GENDER ===")
print(analysis_df.groupby('GENDER')['readmitted_30d'].agg(['mean', 'count']).round(3))

print("\n=== READMISSION RATE BY RACE ===")
print(analysis_df.groupby('RACE')['readmitted_30d'].agg(['mean', 'count']).round(3))

print("\n=== KEY FEATURES BY RACE (mean values) ===")
print(analysis_df.groupby('RACE')[['age_at_admission', 'icu_admission',
                                     'active_conditions', 'length_of_stay']].mean().round(2))

print("\n=== KEY FEATURES BY GENDER (mean values) ===")
print(analysis_df.groupby('GENDER')[['age_at_admission', 'icu_admission',
                                       'active_conditions', 'length_of_stay']].mean().round(2))

=== READMISSION RATE BY GENDER ===
         mean  count
GENDER              
F       0.103    904
M       0.086    758

=== READMISSION RATE BY RACE ===
         mean  count
RACE                
asian   0.149    114
black   0.087    150
native  0.167      6
white   0.091   1392

=== KEY FEATURES BY RACE (mean values) ===
        age_at_admission  icu_admission  active_conditions  length_of_stay
RACE                                                                      
asian              57.34           0.26               9.21           13.66
black              56.18           0.21               7.75           14.23
native             46.82           0.17               6.50           15.32
white              53.77           0.21               8.08           13.83

=== KEY FEATURES BY GENDER (mean values) ===
        age_at_admission  icu_admission  active_conditions  length_of_stay
GENDER                                                                    
F                  54.50       